# Walmart Time Series Forecasting — Feature Engineering

## Objectives

The objective of this notebook is to clean/transform the data and engineering features motivated by `01_eda.ipynb` findings. 

## Outputs

Cleaned, transformed, and scaled dataset saved to `data/processed/` as `X_train.csv`, `X_test.csv`, `y_train.csv`, and `y_test.csv`, ready for modeling in `03_modeling_and_evaluation.ipynb`.

## 2.1 Setup & Imports

Importing `pandas`, same as in `01_eda.ipynb`, as well as `numpy` for log transforms and array operations for lag features. Path constants defined here to ensure reproducibility and consistent file references throughout the notebook.

In [6]:
import numpy as np
import pandas as pd 

PROCESSED_DATA_PATH = '../data/processed/walmart_merged.csv'
X_TRAIN_PATH = '../data/processed/X_train.csv'
X_TEST_PATH = '../data/processed/X_test.csv'
Y_TRAIN_PATH = '../data/processed/y_train.csv'
Y_TEST_PATH = '../data/processed/y_test.csv'

### Load Dataset

In [21]:
df = pd.read_csv(PROCESSED_DATA_PATH)

## 2.2 Data Cleaning
In this section: I handle the negative values found in `Weekly_Sales` and `MarkDown2/3`, Encode for missing values in `MarkDown1-5`, and transform the skewed features. 

### 2.2.1 Handling Negative Values
To determine whether the negative values are real values or just data entry anaomalies, I filter the dataframe to only the negative values for each of the features, then use `.describe()` on the filtered df to check if the negatives are small anomolies or normal-sized values with flipped sign. Then I use .`value_counts().head()` on `Store` and `Dept` to check if the negative values are concentrated in a few stores/departments or evenly spread.

In [14]:
neg_sales = df[df['Weekly_Sales'] < 0]
print(neg_sales['Weekly_Sales'].describe())
print(neg_sales['Store'].value_counts().head())
print(neg_sales['Dept'].value_counts().head())


count    1285.000000
mean      -68.608218
std       231.664245
min     -4988.940000
25%       -41.000000
50%       -13.200000
75%        -4.940000
max        -0.020000
Name: Weekly_Sales, dtype: float64
Store
35    124
18     52
10     50
17     49
15     45
Name: count, dtype: int64
Dept
47    254
18    180
54    146
19     87
94     77
Name: count, dtype: int64


The negative `Weekly_Sales` values are spread across many different `Store`s/`Dept`s. Since there is no dominant cluster, this is likely not an error, but a real signal consistent with returns exceeding sales. 

In [15]:
neg_md2 = df[df['MarkDown2'] < 0]
print(neg_md2['MarkDown2'].describe())
print(neg_md2['Store'].value_counts().head())
print(neg_md2['Dept'].value_counts().head())

count    1311.000000
mean      -30.862517
std        70.446056
min      -265.760000
25%       -10.500000
50%        -7.010000
75%        -2.000000
max        -0.600000
Name: MarkDown2, dtype: float64
Store
41    145
10    142
15    138
9     124
20     72
Name: count, dtype: int64
Dept
1    19
2    19
3    19
4    19
5    19
Name: count, dtype: int64


The negative values for `MarkDown2`, unlike `Weekly_Sales`, has `Dept` `.value_counts()` identical across every department (19) which is highly suspicious since the negative values don't appear to be an independent per-`Dept` event. I will check whether `MarkDown2` is actually measured per-`Dept` or by some other measure. 

To determine if `MarkDown2` is measured per-`Dept` or some other measure, I will take the first `notna()` `MarkDown2` `Store` and `Date`, and check the store-week's mark down value. 

In [8]:
sample = df[df['MarkDown2'].notna()][['Store', 'Date']].iloc[0]
df[(df['Store'] == sample['Store']) & (df['Date'] == sample['Date'])][['Dept', 'MarkDown2']]

,Dept,MarkDown2
92,1,6115.67
235,2,6115.67
378,3,6115.67
521,4,6115.67
664,5,6115.67
...,...,...
9605,94,6115.67
9748,95,6115.67
9870,96,6115.67
10013,97,6115.67


The same `MarkDown2` value (6115.67) is repeated across all `Dept`s for the given store-week. This means that mark down values are recorded at the store-week level and then broadcast onto every `Dept` row, not per department. 

Since the mark down values are recorded at the store-week level, I will check the negative `MarkDown2` values with dropped duplicated.

In [5]:
df[df['MarkDown2'] < 0][['Store', 'Date']].drop_duplicates()

,Store,Date
29629,4,2012-03-23
39922,5,2012-08-17
78788,9,2012-08-10
78792,9,2012-09-07
87634,10,2012-03-16
87652,10,2012-07-20
108027,12,2012-07-06
128207,14,2012-07-13
138253,15,2012-08-24
138262,15,2012-10-26


This shows there are actually only 19 distinct store-week events with negitive mark downs, not 1311 independent rows found earlier. This means the `Dept`-level negative counts were inflated by the broadcasting when dataframes were merged. 

To determine if the negative `MarkDown2` values are real signals or data entry errors, I will check the values nearby, in time, of a store that had a negative `MarkDown2` to see whether those nearby `MarkDown2` values look disconnected from the negative value or if the negative value's magnitude roughly relates to nearby positive value, which would suggest a correction/clawback rather than a data entry error. 

In [ ]:
sample_store = (df[df['Store'] == 4].drop_duplicates(subset=['Date'])).sort_values(by='Date').reset_index()

i_neg = sample_store[sample_store['Date']=='2012-03-23'].index[0]
# look at values nearby
sample_store[['Date','MarkDown2']].iloc[i_neg-5:i_neg+5,]

,Date,MarkDown2
106,2012-02-17,11049.65
107,2012-02-24,4703.88
108,2012-03-02,1394.86
109,2012-03-09,602.02
110,2012-03-16,37.12
111,2012-03-23,-10.50
112,2012-03-30,442.52
113,2012-04-06,NaN
114,2012-04-13,5941.43
115,2012-04-20,4279.41


Since `MarkDown3` likely has the same broadcasting structure, I will skip the full `Store`-`Date` lookup, but will verify with `.describe()` and `.value_counts()` first. 

In [16]:
neg_md3 = df[df['MarkDown3'] < 0]
print(neg_md3['MarkDown3'].describe())
print(neg_md3['Store'].value_counts().head())
print(neg_md3['Dept'].value_counts().head())

count    257.000000
mean      -8.634319
std       12.796205
min      -29.100000
25%      -29.100000
50%       -1.000000
75%       -0.200000
max       -0.200000
Name: MarkDown3, dtype: float64
Store
28    72
31    70
39    69
36    46
Name: count, dtype: int64
Dept
1    4
2    4
3    4
4    4
5    4
Name: count, dtype: int64


Again, `Dept` `.value_counts()` is identical across every department (4), confirming the same broadcast structure as `MarkDown2`. I will not run a separate nearby-values check, since I am extending the `MarkDown2` findings to `MarkDown3` by structural analogy. 

Given the negative values for `Weekly_Sales` and `MarkDown2/3` are real signals, I can't simply remove them since that would result in losing real data, but I still need to transform them since negative values can't be log transformed. Therefore, to deal with the negative values I will multiply the `.sign()` of the values with `.log1p()` of the absolute value, preserving direction while making magnitude log-scale-safe, for `Weekly_Sales` and `MarkDown2/3`.

### 2.2.2 Encoding Missing Values

From the dataset description, the missing values in the mark down columns are not unknown values, but rather instances where no promotion occured. Therefore, I will encode `NaN` values with 0 and add a separate boolean flag column, so that the downstream model can distinguish a true zero (no promotion) from a real value near 0 (small promotion). 

In [ ]:
markdowns = ['MarkDown1', 'MarkDown2', 'MarkDown3', 'MarkDown4', 'MarkDown5']

for m in markdowns:
    # create flag column
    df[f'{m}_No_Promotion'] = df[m].isna().astype(int)
    # encode missing vals with 0
    df[m] = df[m].fillna(0)

print(df.columns.tolist())
# flag count should match how many NaNs existed originally
print(df['MarkDown1_No_Promotion'].sum()) 
print((df['MarkDown1'] == 0).sum())  # should be >= flagged count


['Store', 'Dept', 'Date', 'Weekly_Sales', 'IsHoliday', 'Type', 'Size', 'Temperature', 'Fuel_Price', 'MarkDown1', 'MarkDown2', 'MarkDown3', 'MarkDown4', 'MarkDown5', 'CPI', 'Unemployment', 'MarkDown1_No_Promotion', 'MarkDown2_No_Promotion', 'MarkDown3_No_Promotion', 'MarkDown4_No_Promotion', 'MarkDown5_No_Promotion']
270889
270889


Used `df.columns.tolist()` and compared `MarkDown1_No_Promotion.sum()` and `(df['MarkDown1'] == 0).sum()` which both returned 270889, matching the number of `NaN`s found during EDA, to confirm that the five new flag markdowns were added correctly and the `NaN` values were correctly encoded.

### 2.2.3 Transform Skewed Features

To transform the skewed features (numerically determined which features to transform by absolute value of `.skew() > 0.5`) for down stream linear modeling, I use the expression `np.sign(x) * np.log1p(abs(x))` to deal with negative values, while preserving magnitude. 

In [ ]:
continuous_features = ['Weekly_Sales', 'Size', 'Temperature', 'Fuel_Price', 'MarkDown1', 'MarkDown2', 'MarkDown3', 'MarkDown4', 'MarkDown5', 'CPI', 'Unemployment']
df[continuous_features].skew()

Weekly_Sales     3.262008
Size            -0.325850
Temperature     -0.321404
Fuel_Price      -0.104901
MarkDown1        4.731304
MarkDown2       10.645956
MarkDown3       14.922341
MarkDown4        8.077666
MarkDown5        9.964519
CPI              0.085219
Unemployment     1.183743
dtype: float64

Only `Weekly_Sales`, `MarkDown1-5`, and `Unemployment` have values above the 0.5 threshold, so those will be the only features I transform.

In [24]:
skewed_features = ['Weekly_Sales', 'MarkDown1', 'MarkDown2', 'MarkDown3', 'MarkDown4', 'MarkDown5', 'Unemployment']
for s in skewed_features:
   df[f'{s}_log'] = np.sign(df[s]) * np.log1p(abs(df[s]))

In [27]:
trans_features = ['Weekly_Sales_log', 'MarkDown1_log', 'MarkDown2_log', 'MarkDown3_log', 'MarkDown4_log', 'MarkDown5_log', 'Unemployment_log']
df[trans_features].skew()

Weekly_Sales_log   -1.500279
MarkDown1_log       0.725749
MarkDown2_log       1.736505
MarkDown3_log       2.003662
MarkDown4_log       0.986896
MarkDown5_log       0.640103
Unemployment_log    0.201250
dtype: float64

I save the transformed values into new `_log` columns to preserve the original values for debugging. Either the original or `_log` columns will be dropped during modeling depending on model type. I check transformation by running `.skew()`, which shows that the log transfom reduced skew substantially but did not fully eliminate it for `MarkDown1-5`, and `Weekly_Sales_log` flipped sign — likely due to the small number of large-magnitude negative values combined with a large number of small positive values. Since this transformation is primarily relevant to a baseline linear regression (a random forest is expected to be the stronger model and doesn't require it), I will accept this limitation and move on and will note it in limitations/future-improvemnts section. 

## 2.3 Feature Engineering

### 2.3.1 Calendar Features

### 2.3.2 Lag Features

## 2.4 Temporal Train/Test Split

## 2.5 Context Variable Decision

## 2.6 Export

## 2.7 Decisions Table